In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 119 samples and 89 columns.
First few rows of the data:


,dmfs time,dmfs event,sample_title,gender,sample type,irdm subtype,pam50 subtype,immunity group,proliferation group,er status rna,...,MIA,MYBL2,UBE2C,AURKA,MMP11,SF3A1,CCDC157,APOBEC3G,IL2RG,SH2D1A
0,1.916667,0,SH_TS_BC113,female,Solid tumor,Basal,Basal,iweak,high,ER-,...,8.669655,0.0,10.785072,9.701948,0.0,0.0,0.0,0.0,7.698076,0.000000
1,6.750000,0,SH_TS_BC119,female,Solid tumor,HER2E,HER2E,istrong,high,ER-,...,5.828747,0.0,9.761326,9.029801,0.0,0.0,0.0,0.0,8.699401,6.999987
2,3.416667,0,SH_TS_BC146,female,Solid tumor,Basal,Basal,iweak,high,ER+,...,7.661877,0.0,10.722630,9.294487,0.0,0.0,0.0,0.0,7.882371,5.965590
3,9.750000,0,SH_TS_BC147,female,Solid tumor,LumB,LumB,iweak,high,ER+,...,6.023581,0.0,9.533534,8.878157,0.0,0.0,0.0,0.0,2.705913,7.212216
4,9.083333,0,SH_TS_BC150,female,Solid tumor,LumA,LumA,istrong,low,ER+,...,7.671198,0.0,8.358780,8.501992,0.0,0.0,0.0,0.0,8.094774,7.722999


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for dmfs time or dmfs event: 0
Number of samples with complete data for both variables: 119
Number of censored samples (event_var == 0.0): 105
Number of events (event_var == 1.0): 14
Censored ratio (among complete): 105 / 119 = 0.882 (88.2%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE113863/description.json
